In [33]:
import pandas as pd 
import sqlite3
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
    
con = sqlite3.connect(r'..\data\raw\ipl.db') 
  
def q(sql): 
    """Run a SELECT and show the result as a table.""" 
    return pd.read_sql(sql, con)


def run_sql_file(path): 
    """Read a .sql file and execute every statement in it.""" 
    with open(path, 'r') as f: 
        script = f.read() 
    con.executescript(script) 
    con.commit() 
    print('ran', path)

In [34]:
stats.binomtest(613, 1187, 0.5).pvalue 
# 0.2700  <- the toss result from 4.2, in one line

np.float64(0.2700389712415702)

In [35]:
# Quick check: SciPy is working
check_p = stats.binomtest(613, 1187, 0.5).pvalue
print("Check p-value:", check_p)


Check p-value: 0.2700389712415702


# T1 — Does winning the toss help you win the match?
Finding: Toss winners won 51.64% of decisive matches.
Statistical question: Is the toss winner's match-win rate significantly different from 50%?
Null hypothesis: H₀: Toss winners win 50% of decisive matches.
Test: Binomial test

In [36]:
# T1 — SQL
t1_df = q("""
SELECT
    COUNT(*) AS n,
    SUM(
        CASE
            WHEN toss_winner = match_winner THEN 1
            ELSE 0
        END
    ) AS k
FROM matches_clean
WHERE result = 'win';
""")

display(t1_df)

,n,k
0,1187,613


In [37]:
# T1 — Statistical test
n = int(t1_df.loc[0, "n"])
k = int(t1_df.loc[0, "k"])

observed_rate = k / n
null_rate = 0.50

result = stats.binomtest(k, n, null_rate)
p_value = result.pvalue

ci_low, ci_high = proportion_confint(
    k, n, method="wilson"
)

effect_pp = (observed_rate - null_rate) * 100

print(f"Sample size: {n}")
print(f"Toss-winner wins: {k}")
print(f"Observed rate: {observed_rate*100:.2f}%")
print(f"p-value: {p_value:.6f}")
print(f"Effect: {effect_pp:.2f} percentage points")
print(f"95% CI: {ci_low*100:.2f}% to {ci_high*100:.2f}%")

if p_value < 0.05:
    print("Verdict: Statistically significant evidence of a difference from 50%.")
else:
    print("Verdict: Not enough evidence to conclude that the toss-winner win rate differs from 50%.")

Sample size: 1187
Toss-winner wins: 613
Observed rate: 51.64%
p-value: 0.270039
Effect: 1.64 percentage points
95% CI: 48.80% to 54.48%
Verdict: Not enough evidence to conclude that the toss-winner win rate differs from 50%.


# T2 — Does the decision taken after the toss matter?
Finding: Toss winners who chose to field had a different win rate from those who chose to bat.
Statistical question: Is toss decision associated with whether the toss winner wins the match?
Null hypothesis: H₀: Toss decision and toss-winner match outcome are independent.
Test: Chi-square test

In [38]:
# T2 — SQL
t2_df = q("""
SELECT
    toss_decision,
    SUM(
        CASE
            WHEN toss_winner = match_winner THEN 1
            ELSE 0
        END
    ) AS won,
    SUM(
        CASE
            WHEN toss_winner <> match_winner THEN 1
            ELSE 0
        END
    ) AS lost
FROM matches_clean
WHERE result = 'win'
GROUP BY toss_decision;
""")

display(t2_df)

,toss_decision,won,lost
0,bat,184,218
1,field,429,356


In [39]:
# T2 — Chi-square + Cramer's V
table = t2_df[["won", "lost"]].to_numpy()

chi2, p_value, dof, expected = stats.chi2_contingency(table)

n = table.sum()
cramers_v = np.sqrt(chi2 / n)

print(f"Chi-square: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.6f}")
print(f"Sample size: {n}")
print(f"Cramer's V: {cramers_v:.4f}")

if p_value < 0.05:
    print("Verdict: Evidence of an association between toss decision and toss-winner outcome.")
else:
    print("Verdict: Not enough evidence of an association between toss decision and toss-winner outcome.")

Chi-square: 8.0401
Degrees of freedom: 1
p-value: 0.004575
Sample size: 1187
Cramer's V: 0.0823
Verdict: Evidence of an association between toss decision and toss-winner outcome.


# T3 — Does the chasing team really have an advantage?

**Finding:** Chasing teams won 54.51% of decisive matches.

**Statistical question:** Is the chasing win rate significantly different from 50%?

**Null hypothesis:** H₀: Chasing teams win 50% of matches.

**Test:** Binomial test


In [40]:
# T3 — SQL
t3_df = q("""
SELECT
    COUNT(*) AS n,
    SUM(chase_won) AS k
FROM v_match_totals;
""")

display(t3_df)

,n,k
0,1187,647


In [41]:
# T3 — Statistical test
n = int(t3_df.loc[0, "n"])
k = int(t3_df.loc[0, "k"])

observed_rate = k / n
null_rate = 0.50

result = stats.binomtest(k, n, null_rate)
p_value = result.pvalue

ci_low, ci_high = proportion_confint(
    k, n, method="wilson"
)

effect_pp = (observed_rate - null_rate) * 100

print(f"Sample size: {n}")
print(f"Chases won: {k}")
print(f"Observed chase win rate: {observed_rate*100:.2f}%")
print(f"p-value: {p_value:.6f}")
print(f"Effect: {effect_pp:.2f} percentage points")
print(f"95% CI: {ci_low*100:.2f}% to {ci_high*100:.2f}%")

if p_value < 0.05:
    print("Verdict: Evidence that the chase win rate differs from 50%.")
else:
    print("Verdict: Not enough evidence to conclude that the chase win rate differs from 50%.")

Sample size: 1187
Chases won: 647
Observed chase win rate: 54.51%
p-value: 0.002080
Effect: 4.51 percentage points
95% CI: 51.66% to 57.32%
Verdict: Evidence that the chase win rate differs from 50%.


# T4 — Does the chase win rate fall as the target rises?

**Finding:** Chase success changes across target bands.

**Statistical question:** Is chase outcome associated with the target band?

**Null hypothesis:** H₀: Chase outcome is independent of target band.

**Test:** Chi-square test

In [42]:
# T4 — SQL
t4_df = q("""
SELECT
    CASE
        WHEN first_innings_runs < 140 THEN 'Under 140'
        WHEN first_innings_runs < 160 THEN '140-159'
        WHEN first_innings_runs < 180 THEN '160-179'
        WHEN first_innings_runs < 200 THEN '180-199'
        ELSE '200+'
    END AS band,
    SUM(chase_won) AS won,
    SUM(1 - chase_won) AS lost
FROM v_match_totals
GROUP BY band;
""")

display(t4_df)

,band,won,lost
0,140-159,171,70
1,160-179,166,146
2,180-199,83,134
3,200+,46,152
4,Under 140,181,38


In [43]:
# T4 — Chi-square + Cramer's V
table = t4_df[["won", "lost"]].to_numpy()

chi2, p_value, dof, expected = stats.chi2_contingency(table)

n = table.sum()
cramers_v = np.sqrt(chi2 / n)

t4_display = t4_df.copy()
t4_display["chase_win_rate"] = (
    t4_display["won"] /
    (t4_display["won"] + t4_display["lost"]) * 100
)

display(t4_display)

print(f"Chi-square: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"p-value: {p_value:.6e}")
print(f"Sample size: {n}")
print(f"Cramer's V: {cramers_v:.4f}")

if p_value < 0.05:
    print("Verdict: Evidence that chase outcome differs across target bands.")
else:
    print("Verdict: Not enough evidence of a difference across target bands.")

,band,won,lost,chase_win_rate
0,140-159,171,70,70.954357
1,160-179,166,146,53.205128
2,180-199,83,134,38.248848
3,200+,46,152,23.232323
4,Under 140,181,38,82.648402


Chi-square: 197.6789
Degrees of freedom: 4
p-value: 1.185422e-41
Sample size: 1187
Cramer's V: 0.4081
Verdict: Evidence that chase outcome differs across target bands.
